<a href="https://colab.research.google.com/github/pascal-maker/agents/blob/main/ragapplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain_openai langchain_community

In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain import hub

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = ""


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load data from a web page
loader = WebBaseLoader("https://pascal-maker.github.io/developedbypascalmusabyimana/")
data = loader.load()

# Split into chunks for indexing
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)

In [ ]:
# Initialize embeddings using OpenAI
embeddings = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_splits, embedding=embeddings)


In [ ]:
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 51.2 MB/s eta 0:00:00


In [ ]:
retriever = vectorstore.as_retriever()


In [ ]:
# Pull a RAG prompt from the LangChain hub
prompt = hub.pull("rlm/rag-prompt")


/usr/local/lib/python3.11/dist-packages/langsmith/client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
# Initialize the language model
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableParallel(
        context=retriever | format_docs,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
)


In [ ]:
# Ask a question
question = "What is the main idea of the data source?"
answer = rag_chain.invoke(question)

# Print the answer
print(answer)


content='The main idea of the data source is that the developer, Pascal Musabyimana, is a Full-Stack Developer with 3 years of experience in software development, specializing in web and mobile applications and computer-vision projects. He is proficient in various languages and frameworks such as React, Laravel, React-Native, OpenCV, Ultralytics, YOLOv8, Objective-C, Java, JavaScript, Firebase, and Swift. Pascal is available for freelance work and aims to help entrepreneurs fulfill their dreams by creating websites and apps.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 396, 'total_tokens': 505, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-9ebca8d